In [ ]:
!pip install -qU  unsloth
!pip install -qU  trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 19.3 MB/s eta 0:00:00


In [ ]:
# !pip uninstall -y trl

In [ ]:
# !pip cache purge

In [ ]:
import pandas as pd
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer, TrainingArguments
import torch
from datasets import Dataset, load_dataset
import json
import re
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass
import wandb
from peft import PeftModel
from sklearn.utils import resample

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import torch._dynamo
torch._dynamo.config.cache_size_limit = 64

In [ ]:
@dataclass
class EvaluationScores:
    status_correctness: int = 0
    score_accuracy: float = 0.0
    skills_matching: int = 0
    experience_evaluation: int = 0


class GRPOResumeRewardFunction:
    def __init__(self, weights: Dict[str, float] = None):
        self.default_weights = {
            'status_correctness': 0.4,
            'score_accuracy': 0.2,
            'skills_matching': 0.2,
            'experience_evaluation': 0.2,
        }
        self.weights = weights if weights is not None else self.default_weights
        total_weight = sum(self.weights.values())
        if total_weight > 0:
            self.weights = {k: v / total_weight for k, v in self.weights.items()}

    def extract_model_output(self, completion: str) -> Dict[str, Any]:
        try:
            start_index = completion.find('{')
            end_index = completion.rfind('}')
            if start_index != -1 and end_index != -1 and end_index > start_index:
                json_str = completion[start_index: end_index + 1]
                data = json.loads(json_str)
                return {
                    'model_score': int(data.get("score", 0)),
                    'model_status': str(data.get("status", "REJECTED")).upper()
                }
        except (json.JSONDecodeError, ValueError, TypeError):
            pass
        return {'model_score': 0, 'model_status': 'REJECTED'}

    def evaluate_status_correctness(self, model_status: str, expected_status: str) -> int:
        """
        Uses the 'gentle polishing' penalties that we know lead to stable training.
        """
        if model_status in ["APPROVE", "CONSIDER", "STRONGLY_CONSIDER", "SELECTED"]:
            model_status = "SELECTED"

        # --- CORRECTED: Using softer, more stable penalties ---
        if model_status == "SELECTED" and expected_status == "SELECTED":
            return 2  # High reward for True Positives
        elif model_status == "REJECTED" and expected_status == "SELECTED":
            return -2 # Gentle penalty for False Negatives (was -5)
        elif model_status == "REJECTED" and expected_status == "REJECTED":
            return 0  # No reward or penalty for True Negatives
        else: # False Positive
            return -1 # Gentle penalty for False Positives (was -2)

    def evaluate_score_accuracy(self, model_score: int, ground_truth_score: Optional[int]) -> float:
        if ground_truth_score is None:
            return 0.0
        error = abs(model_score - ground_truth_score)
        reward = max(-1.0, 1.0 - (error / 50.0))  # Continuous reward from 1.0 down to -1.0
        return reward

    def extract_skills_from_data(self, job_skills_str: str, evaluation_json_str: str) -> Tuple[List[str], List[str]]:
        job_skills = []
        candidate_skills = []

        if job_skills_str:
            job_skills = [skill.strip().lower() for skill in job_skills_str.split(',')]

        try:
            data = json.loads(evaluation_json_str)
            present_skills = data.get('skills_match', {}).get('present_skills', [])
            candidate_skills = [skill.strip().lower() for skill in present_skills]
        except Exception:
            pass

        return job_skills, candidate_skills

    def evaluate_skills_matching(self, job_skills: List[str], candidate_skills: List[str]) -> int:
        if not job_skills or not candidate_skills:
            return -1
        match_count = len(set(job_skills) & set(candidate_skills))
        match_ratio = match_count / len(job_skills)
        if match_ratio >= 0.75:
            return 1
        elif match_ratio >= 0.4:
            return 0
        else:
            return -1

    def evaluate_experience_evaluation(self, model_score: int, evaluation_json_str: str) -> int:
        years = 0
        try:
            data = json.loads(evaluation_json_str)
            exp_data = data.get('experience_relevance', {})
            explanation = exp_data.get('explanation', '').lower()

            # A simple heuristic to infer experience level
            if '10' in explanation or 'senior' in explanation:
                years = 10
            elif '5' in explanation or 'mid' in explanation:
                years = 5
            elif '3' in explanation:
                years = 3
            elif '1' in explanation or 'junior' in explanation:
                years = 1
            elif 'no' in explanation or 'entry' in explanation:
                years = 0
        except Exception:
            return 0  # Neutral reward if parsing fails

        # Check if the model's predicted score is logically consistent with the experience level
        if years >= 7 and model_score >= 75:
            return 1
        elif 3 <= years <= 6 and 50 <= model_score < 80:
            return 1
        elif years <= 2 and model_score < 50:
            return 1
        # Penalize clear inconsistencies
        elif model_score > 85 and years <= 2:
            return -1
        elif model_score < 40 and years >= 5:
            return -1
        else:
            return 0

    def calculate_single_reward(self, completion: str, ground_truth_data: Dict[str, Any]) -> float:
        parsed_output = self.extract_model_output(completion)
        try:
            evaluation_json = ground_truth_data.get('evaluation_json', '{}')
            ground_truth_score = json.loads(evaluation_json).get('overall_score')
        except (json.JSONDecodeError, TypeError):
            ground_truth_score = None
            evaluation_json = '{}'

        ground_truth_status = ground_truth_data.get('status', '')
        expected_status = 'SELECTED' if ground_truth_status in ['APPROVE', 'CONSIDER', 'STRONGLY_CONSIDER'] else 'REJECTED'
        job_skills_str = ground_truth_data.get('job_description_skills', '')

        job_skills, candidate_skills = self.extract_skills_from_data(job_skills_str, evaluation_json)

        scores = EvaluationScores()
        scores.status_correctness = self.evaluate_status_correctness(parsed_output['model_status'], expected_status)
        scores.score_accuracy = self.evaluate_score_accuracy(parsed_output['model_score'], ground_truth_score)
        scores.skills_matching = self.evaluate_skills_matching(job_skills, candidate_skills)
        scores.experience_evaluation = self.evaluate_experience_evaluation(parsed_output['model_score'], evaluation_json)

        parameter_scores = {
            'status_correctness': scores.status_correctness,
            'score_accuracy': scores.score_accuracy,
            'skills_matching': scores.skills_matching,
            'experience_evaluation': scores.experience_evaluation
        }

        final_reward = sum(
            self.weights[param] * score
            for param, score in parameter_scores.items()
            if param in self.weights
        )
        return final_reward


def grpo_compatible_reward_function(prompts: List[str], completions: List[str], **kwargs) -> List[float]:
    reward_function = GRPOResumeRewardFunction()
    evaluation_jsons = kwargs.get('evaluation_json', [])
    statuses = kwargs.get('status', [])
    job_description_skills = kwargs.get('job_description_skills', [])
    rewards = []

    for i, completion in enumerate(completions):
        try:
            ground_truth_data = {
                'evaluation_json': evaluation_jsons[i] if i < len(evaluation_jsons) else '{}',
                'status': statuses[i] if i < len(statuses) else '',
                'job_description_skills': job_description_skills[i] if i < len(job_description_skills) else ''
            }
            reward = reward_function.calculate_single_reward(completion, ground_truth_data)
            rewards.append(float(reward))
        except Exception:
            rewards.append(0.0)

    return rewards


In [ ]:
def create_prompt(job_skills, candidate_skills):
    return (
        f"You are an HR expert. Evaluate the candidate's skills against the job requirements.\n\n"
        f"### Job Requirements:\n{job_skills}\n\n"
        f"### Candidate Skills:\n{candidate_skills}\n\n"
        f"Provide your assessment in a simple JSON format containing only the status and score."
    )

def prepare_dataset(df):
    dataset_samples = []
    for _, item in df.iterrows():
        item_dict = item.to_dict()
        job_skills = item_dict.get('job_description_skills', '')

        candidate_skills_str = ''
        eval_json_str = item_dict.get('evaluation_json', '{}')
        if eval_json_str:
            try:
                eval_data = json.loads(eval_json_str)
                present_skills = eval_data.get('skills_match', {}).get('present_skills', [])
                candidate_skills_str = ", ".join(present_skills)
            except (json.JSONDecodeError, TypeError): pass

        sample = {
            'prompt': create_prompt(job_skills, candidate_skills_str),
            'evaluation_json': eval_json_str,
            'status': item_dict.get('status', 'REJECT'),
            'job_description_skills': job_skills
        }
        dataset_samples.append(sample)
    return Dataset.from_list(dataset_samples)

In [ ]:
full_data_path = "/content/o6ai-agent-hr/data/resume_evaluation_dataset-3000.json"
print(f"Loading full dataset from {full_data_path}...")
full_df = pd.read_json(full_data_path)
full_df = full_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 90/10 split
train_size = int(0.9 * len(full_df))
df_train = full_df[:train_size]
df_eval = full_df[train_size:]

# Print Data Distribution
print("\n--- Data Distribution ---")
df_train['simple_status'] = df_train['status'].apply(lambda x: 'SELECTED' if x in ['APPROVE', 'CONSIDER', 'STRONGLY_CONSIDER'] else 'REJECTED')
print(f"Training set size: {len(df_train)}")
print("Training set distribution:")
print(df_train['simple_status'].value_counts())

df_eval['simple_status'] = df_eval['status'].apply(lambda x: 'SELECTED' if x in ['APPROVE', 'CONSIDER', 'STRONGLY_CONSIDER'] else 'REJECTED')
print(f"\nEvaluation set size: {len(df_eval)}")
print("Evaluation set distribution:")
print(df_eval['simple_status'].value_counts())
print("-" * 25)


Loading full dataset from /content/o6ai-agent-hr/data/resume_evaluation_dataset-3000.json...

--- Data Distribution ---
Training set size: 2700
Training set distribution:
simple_status
SELECTED    1359
REJECTED    1341
Name: count, dtype: int64

Evaluation set size: 300
Evaluation set distribution:
simple_status
REJECTED    159
SELECTED    141
Name: count, dtype: int64
-------------------------


/tmp/ipython-input-2852974727.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train['simple_status'] = df_train['status'].apply(lambda x: 'SELECTED' if x in ['APPROVE', 'CONSIDER', 'STRONGLY_CONSIDER'] else 'REJECTED')
/tmp/ipython-input-2852974727.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_eval['simple_status'] = df_eval['status'].apply(lambda x: 'SELECTED' if x in ['APPROVE', 'CONSIDER', 'STRONGLY_CONSIDER'] else 'REJECTED')


In [ ]:
# Prepare datasets for the trainer
train_dataset = prepare_dataset(df_train)
eval_dataset = prepare_dataset(df_eval)

In [ ]:
wandb.login()
print("\n--- [GRPO Final] Starting Final GRPO Training Run ---")
base_model_name = "unsloth/Qwen2-0.5B-Instruct-bnb-4bit"
sft_adapter_path = "/content/o6ai-agent-hr/models/sft_v4_large_balanced_model/checkpoint-676"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name, max_seq_length=2048, dtype=None, load_in_4bit=True
)

# Apply a fresh PEFT config to the base model to make it trainable
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=3407,
)

# Now, load the SFT weights into the trainable adapter
print(f"Loading SFT adapter weights from: {sft_adapter_path}")
model.load_adapter(sft_adapter_path, adapter_name="default")

model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shreyanshjaino6ai (o6ailabs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth 2025.9.9 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Loading SFT adapter weights from: /content/o6ai-agent-hr/models/sft_v4_large_balanced_model/checkpoint-676
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [ ]:
# --- Training Arguments (FULLY CORRECTED & UPDATED) ---
# This block now includes all your specific evaluation and saving strategies,
# combined with the gentle polishing parameters from the prompt.
output_dir_grpo = "/content/o6ai-agent-hr/models/grpo_final_model-v1"
grpo_training_args = GRPOConfig(
    # --- Core Polishing Parameters ---
    learning_rate=2e-6,
    num_train_epochs=1,

    # --- Speed & Efficiency Optimizations ---
    torch_compile=True,
    torch_compile_mode="reduce-overhead",
    gradient_checkpointing=False, # ✅ Faster if your VRAM allows it
    fp16=False,
    optim="paged_adamw_8bit",
    # bp16 = True,

    # --- Corrected Saving & Evaluation Strategy ---
    save_strategy="epoch",        # ✅ More efficient: saves only once at the very end
    eval_strategy="no",     # ✅ Correctly avoids the evaluation bug
    load_best_model_at_end=False, # ❗ CRITICAL FIX: Must be False when there's no evaluation

    # --- Batching & GRPO Parameters ---
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,   # This argument will be ignored
    gradient_accumulation_steps=4,
    beta=0.1,
    num_generations=2,

    # --- Logging & Housekeeping ---
    output_dir=output_dir_grpo,
    logging_dir=f"{output_dir_grpo}/logs",
    logging_steps=10,
    report_to="wandb",
    remove_unused_columns=False,
)

# --- Initialize Trainer (CORRECTED) ---
grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_training_args, # Use the GRPOConfig object here
    train_dataset=train_dataset,
    # eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    reward_funcs=grpo_compatible_reward_function,
)

In [ ]:
wandb.init(project="o6ai-resume-evaluation-GRPO", name="grpo-final-run-on-sft-v4", reinit=True)

# --- Start Training ---
print("\n--- [GRPO Final] Starting training... ---")
grpo_trainer.train()

print("\n✅ [GRPO Final] Training completed successfully!")
grpo_trainer.save_model()
print(f"✅ [GRPO Final] Final model saved to: {output_dir_grpo}")
wandb.finish()


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



--- [GRPO Final] Starting training... ---


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,700 | Num Epochs = 1 | Total steps = 337
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Unsloth: Enabled auto compiling


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / grpo_compatible_reward_function / mean,rewards / grpo_compatible_reward_function / std
10,4.938000,-0.336200,0.000000,1.968750,1.000000,16.500000,0.000000,1.968750,1.000000,16.500000,0,0,0,0,0,12.345076,-0.336200,0.365040
20,2.846900,-0.262450,0.000000,1.356250,1.000000,6.700000,0.000000,1.356250,1.000000,6.700000,No Log,No Log,No Log,No Log,No Log,7.117160,-0.262450,0.354150
30,3.216600,-0.222150,0.031183,12.831250,1.000000,128.800000,0.037500,3.362500,1.000000,37.300000,No Log,No Log,No Log,No Log,No Log,8.041464,-0.222150,0.400350
40,0.214300,-0.156900,0.165675,33.318750,1.000000,196.200000,0.062500,18.400358,1.000000,118.800000,No Log,No Log,No Log,No Log,No Log,0.535668,-0.156900,0.468989
50,0.077000,-0.149675,0.158215,46.725000,1.000000,215.200000,0.087500,26.650477,1.000000,150.600000,No Log,No Log,No Log,No Log,No Log,0.192471,-0.149675,0.476475
60,0.080800,-0.117575,0.140750,60.750000,1.000000,229.600000,0.131250,31.237520,1.000000,139.800000,No Log,No Log,No Log,No Log,No Log,0.201960,-0.117575,0.479518
70,0.081200,-0.038625,0.248796,80.625000,1.000000,256.000000,0.212500,33.927907,1.000000,150.700000,No Log,No Log,No Log,No Log,No Log,0.203119,-0.038625,0.510378
80,0.099000,-0.118400,0.218142,84.637500,1.000000,256.000000,0.181250,47.520732,1.000000,169.500000,No Log,No Log,No Log,No Log,No Log,0.247411,-0.118400,0.493162
90,0.094300,-0.151325,0.170590,90.537500,1.000000,256.000000,0.237500,37.886851,1.000000,153.300000,No Log,No Log,No Log,No Log,No Log,0.235851,-0.151325,0.494251
100,0.100200,-0.122125,0.239603,86.193750,1.000000,256.000000,0.193750,45.225210,1.000000,168.800000,No Log,No Log,No Log,No Log,No Log,0.250546,-0.122125,0.517447


Unsloth: Will smartly offload gradients to save VRAM!

✅ [GRPO Final] Training completed successfully!
✅ [GRPO Final] Final model saved to: /content/o6ai-agent-hr/models/grpo_final_model-v1


profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▃█▁▂▂▂▂▂▃▃▂▃▃▁▂▁▂▂▄▂▁▃▂▂▃▅▂▂▄▃▂▂▁▂▃▂▁▂▃▂
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▃▁▁▁▁▁█▁▁▁██▁▁▁▇█▁▁▁█▆▁▁▁▁██▁▁██▁▁▁▁▁▁█▁
profiling/Time taken: UnslothGRPOTrainer.grpo_compatible_reward_function,▂▁▁▁▂▃▆▅▃▄▄▅▄▆▅▃▄▆▃▇▄▄▂▅▅▅▅▅█▆▅▄▃▅▄▃▄▄▅▅
profiling/Time taken: UnslothGRPOTrainer.transformers.generate,▁▁▁▃▁▇██▄███████████████████████████████
train/completion_length,▁▁▄▆▇▇██████▇█████████████████▇███
train/completions/clipped_ratio,▁▁▂▃▄▅▇▆█▇▆▅▅▅▅▅▇▅▅▆▅▆▆▆▆▅▆▆▅▅▅▆▅▄
train/completions/max_length,▁▁▄▆▇▇██████▇█████████████████▇███
train/completions/max_terminated_length,▁▁▂▅▆▆▆▇▆▇▅▇▆▆▇█▆█▇▇▆▇▅▆▇▅▇▆▆█▇▇▇▇
train/completions/mean_length,▁▁▂▄▅▆▇███▆▇▆▆▆▆▇▆▆▇▆▇▆▆▇▆▆▆▆▆▇▇▇▇
train/completions/mean_terminated_length,▁▁▁▃▅▅▆█▆▇▅▇▆▆▇▇▆▇▇▇▅▇▅▆█▆▆▆▆▇▇▇██
+14,...
